# Databricks PyNNLF Smoke Test

This notebook checks whether the publication PyNNLF workflow can run inside Databricks without editing paths. It discovers the repo/project location automatically, imports PyNNLF from the local `src/` folder, validates one small publication dataset, and runs a small smoke-test experiment into an isolated output folder.

Default run mode is `fast`: `ds22`, `fh8`, `m1_naive_hp1`, and `m6_lr_hp1`. This is intentionally small so you can quickly test Databricks path handling, dependency installation, CSV reads, model execution, and result writing. On Databricks, the notebook installs PyNNLF from this local repo before importing it.


## 1. Discover Repo And Publication Paths

This cell works both locally and in Databricks Repos/Workspace files. It searches upward from the current working directory and, when available, the Databricks notebook path.


In [ ]:
from pathlib import Path
import os
import sys


def databricks_notebook_candidates():
    candidates = []
    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        nb_path = ctx.notebookPath().get()
    except Exception:
        return candidates

    # Databricks notebook paths often look like /Repos/user/repo/path/to/notebook.
    # The corresponding driver filesystem path is commonly /Workspace/Repos/...
    raw = Path(nb_path)
    candidates.append(raw)
    if nb_path.startswith('/Workspace/'):
        candidates.append(Path(nb_path))
    else:
        candidates.append(Path('/Workspace') / nb_path.lstrip('/'))
    return candidates


def upward_candidates(start):
    start = Path(start).resolve()
    return [start, *start.parents]


def find_repo_root():
    starts = [Path.cwd(), *databricks_notebook_candidates()]
    seen = set()
    candidates = []
    for start in starts:
        try:
            for candidate in upward_candidates(start):
                key = str(candidate)
                if key not in seen:
                    seen.add(key)
                    candidates.append(candidate)
        except Exception:
            continue

    for candidate in candidates:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'pynnlf').exists():
            return candidate

    details = '\n'.join(str(c) for c in candidates[:30])
    raise FileNotFoundError(
        'Could not locate the PyNNLF repo root. Checked candidates including:\n' + details
    )


REPO_ROOT = find_repo_root()
PROJECT_DIR = REPO_ROOT / 'publication' / 'journal_article_1'

if not PROJECT_DIR.exists():
    raise FileNotFoundError(f'Publication project folder not found: {PROJECT_DIR}')

os.chdir(PROJECT_DIR)
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

print(f'Repo root: {REPO_ROOT}')
print(f'Publication project: {PROJECT_DIR}')
print(f'Current working directory: {Path.cwd()}')


## 2. Install PyNNLF And Smoke-Test Dependencies

On Databricks serverless, the base Python environment may not include PyNNLF dependencies such as `dill` and `PyYAML`. This cell installs PyNNLF from the local repo, not GitHub, so it uses the exact publication code/data/specs currently in your workspace.

The local install uses `--ignore-requires-python --no-deps` because some serverless environments use Python 3.10 while this repo declares Python 3.11+. Then the notebook installs only the dependencies needed for the selected smoke profile, using pinned versions where Databricks serverless is known to be sensitive. In particular, `typing_extensions==4.12.2` is pinned so Torch can import cleanly. The default profile is `minimal`, which is enough for naive and linear regression.


In [ ]:

import importlib
import subprocess
import sys


def running_on_databricks():
    if 'DATABRICKS_RUNTIME_VERSION' in os.environ:
        return True
    try:
        dbutils  # noqa: F821
        return True
    except Exception:
        return False


def get_install_profile(default='minimal'):
    choices = ['minimal', 'xgb', 'torch', 'prophet', 'full']
    try:
        dbutils.widgets.dropdown('install_profile', default, choices, 'Install profile')  # noqa: F821
        value = dbutils.widgets.get('install_profile')  # noqa: F821
    except Exception:
        value = default
    if value not in choices:
        raise ValueError(f'Unsupported install_profile={value!r}; expected one of {choices}')
    return value


def pip_install(args):
    cmd = [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', *args]
    print('Running:', ' '.join(str(part) for part in cmd))
    subprocess.check_call(cmd)


INSTALL_PROFILE = get_install_profile()
print(f'Install profile: {INSTALL_PROFILE}')

if running_on_databricks():
    print('Databricks environment detected. Installing PyNNLF from local repo...')
    pip_install(['--ignore-requires-python', '--no-deps', '-e', str(REPO_ROOT)])

    # Versions are pinned where Databricks serverless is likely to keep an incompatible
    # preinstalled package. These pins are based on the old dev-db requirements file
    # and the current repo pyproject. The key Torch fix is typing_extensions==4.12.2.
    deps = [
        'typing_extensions==4.12.2',
        'PyYAML==6.0.2',
        'dill==0.3.9',
        'numpy==2.1.1',
        'pandas==2.2.3',
        'matplotlib==3.9.2',
        'scikit-learn==1.6.0',
        'scipy==1.14.1',
        'statsmodels==0.14.4',
        'joblib==1.4.2',
    ]
    if INSTALL_PROFILE in {'xgb', 'full'}:
        deps += ['xgboost==3.0.0']
    if INSTALL_PROFILE in {'torch', 'full'}:
        deps += ['torch==2.5.1']
    if INSTALL_PROFILE in {'prophet', 'full'}:
        deps += ['prophet==1.1.6', 'cmdstanpy==1.2.5', 'holidays==0.72']

    # --upgrade makes the notebook replace stale Databricks-preinstalled packages
    # such as old typing_extensions, which can break torch imports.
    pip_install(['--upgrade', *deps])
    importlib.invalidate_caches()

    print('Install complete. Strongly recommended on Databricks: restart Python, then rerun from the top.')
    try:
        print('Restart command available: dbutils.library.restartPython()')
    except Exception:
        pass
else:
    print('Not running on Databricks; skipping automatic pip install.')


## 3. Check Python Environment

This checks the packages needed for the default `fast` smoke test. It also reports optional packages used by the heavier paper models.


In [ ]:
import importlib
import platform

print(f'Python: {sys.version}')
print(f'Platform: {platform.platform()}')

required_for_fast = ['numpy', 'pandas', 'sklearn', 'yaml']
optional_for_paper = ['xgboost', 'torch', 'prophet', 'statsmodels']


def import_status(module_name):
    try:
        module = importlib.import_module(module_name)
        version = getattr(module, '__version__', 'version unavailable')
        return 'OK', version
    except Exception as exc:
        return 'MISSING', repr(exc)

print('\nRequired for default fast smoke test:')
for module_name in required_for_fast:
    status, detail = import_status(module_name)
    print(f'  {module_name}: {status} ({detail})')

print('\nOptional/heavier paper dependencies:')
for module_name in optional_for_paper:
    status, detail = import_status(module_name)
    print(f'  {module_name}: {status} ({detail})')

import pynnlf
print(f'\nPyNNLF import OK: {pynnlf.__version__}')


## 4. Validate A Small Publication Dataset

The default smoke test uses `ds22`, the clean 44-household SA BESS underlying-load dataset. This is small enough for a quick Databricks test and belongs to the unfinished paper experiment block.


In [ ]:
import pandas as pd

DATASET_ID = 'ds22'
DATA_FILE = PROJECT_DIR / 'data' / 'ds22_sa_bess_44hh_pos_underlying_load_30min.csv'

if not DATA_FILE.exists():
    raise FileNotFoundError(f'Missing smoke-test dataset: {DATA_FILE}')

df = pd.read_csv(DATA_FILE, parse_dates=['datetime'])
required_columns = {'datetime', 'netload_kW'}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f'{DATA_FILE.name} is missing required columns: {sorted(missing_columns)}')

summary = {
    'rows': len(df),
    'start': df['datetime'].min(),
    'end': df['datetime'].max(),
    'duplicate_timestamps': int(df['datetime'].duplicated().sum()),
    'missing_netload_kW': int(df['netload_kW'].isna().sum()),
    'mean_netload_kW': float(df['netload_kW'].mean()),
    'min_netload_kW': float(df['netload_kW'].min()),
    'max_netload_kW': float(df['netload_kW'].max()),
}
summary


## 5. Choose Smoke-Test Mode

On Databricks, use the widget at the top of the notebook if you want a heavier test. If no widget system is available, the notebook uses `fast`.

- `fast`: naive + linear regression; quick path/dependency test.
- `xgb`: adds XGBoost.
- `torch`: adds LSTM to test PyTorch.
- `prophet`: adds Prophet/CmdStanPy.
- `all_smoke`: runs all of the above.


In [ ]:
def get_smoke_mode(default='fast'):
    choices = ['fast', 'xgb', 'torch', 'prophet', 'all_smoke']
    try:
        dbutils.widgets.dropdown('smoke_mode', default, choices, 'Smoke test mode')
        value = dbutils.widgets.get('smoke_mode')
    except Exception:
        value = default
    if value not in choices:
        raise ValueError(f'Unsupported smoke_mode={value!r}; expected one of {choices}')
    return value


SMOKE_MODE = get_smoke_mode()
MODEL_SETS = {
    'fast': [('m1', 'hp1'), ('m6', 'hp1')],
    'xgb': [('m1', 'hp1'), ('m6', 'hp1'), ('m17', 'hp1')],
    'torch': [('m1', 'hp1'), ('m6', 'hp1'), ('m13', 'hp2')],
    'prophet': [('m1', 'hp1'), ('m6', 'hp1'), ('m16', 'hp1')],
    'all_smoke': [('m1', 'hp1'), ('m6', 'hp1'), ('m17', 'hp1'), ('m13', 'hp2'), ('m16', 'hp1')],
}

SMOKE_MODELS = MODEL_SETS[SMOKE_MODE]
FORECAST_HORIZON_ID = 'fh8'
print(f'Smoke mode: {SMOKE_MODE}')
print(f'Dataset: {DATASET_ID}')
print(f'Forecast horizon: {FORECAST_HORIZON_ID}')
print(f'Models: {SMOKE_MODELS}')


## 6. Run Smoke Experiments In An Isolated Output Folder

Results are written under `results/00_data_exploration_and_processing/03_databricks_test/experiment_result_smoke`, not the paper's main `experiment_result` folder. Completed smoke runs are skipped if you rerun the notebook.


In [ ]:
import yaml
from pynnlf.discovery import discover_dataset_path, discover_model_name
from pynnlf.engine import run_experiment_engine
from pynnlf.hyperparams import load_hyperparameters, get_hp

CONFIG_PATH = PROJECT_DIR / 'specs' / 'pynnlf_config.yaml'
HYPERPARAMS_PATH = PROJECT_DIR / 'models' / 'hyperparameters.yaml'
MODELS_DIR = PROJECT_DIR / 'models'
DATA_DIR = PROJECT_DIR / 'data'
SMOKE_ROOT = PROJECT_DIR / 'results' / '00_data_exploration_and_processing' / '03_databricks_test'
SMOKE_RESULTS_DIR = SMOKE_ROOT / 'experiment_result_smoke'
SMOKE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
config.setdefault('plot', {})['enabled'] = False
hparams = load_hyperparameters(HYPERPARAMS_PATH)
forecast_horizon_min = int(config['forecast_horizons'][FORECAST_HORIZON_ID])
dataset_path = discover_dataset_path(DATA_DIR, DATASET_ID)


def completed_keys(results_root):
    keys = set()
    for result_file in sorted(Path(results_root).glob('E*/E*_a1_experiment_result.csv')):
        try:
            row = pd.read_csv(result_file, nrows=1).iloc[0]
            keys.add((
                str(row.get('dataset_no', '')),
                int(row.get('forecast_horizon_min')),
                str(row.get('model_no', '')),
                str(row.get('hyperparameter_no', '')),
            ))
        except Exception:
            continue
    return keys


done = completed_keys(SMOKE_RESULTS_DIR)
for model_id, hp_no in SMOKE_MODELS:
    key = (DATASET_ID, forecast_horizon_min, model_id, hp_no)
    if key in done:
        print(f'[skip] {DATASET_ID} {FORECAST_HORIZON_ID} {model_id} {hp_no}')
        continue

    model_name = discover_model_name(MODELS_DIR, model_id)
    hp = get_hp(hparams, model_name, hp_no)
    print(f'[run] {DATASET_ID} {FORECAST_HORIZON_ID} {model_id} {hp_no} -> {model_name}')
    run_experiment_engine(
        dataset_path=dataset_path,
        forecast_horizon_min=forecast_horizon_min,
        model_name=model_name,
        hyperparameter_no=hp_no,
        hyperparameter=hp,
        output_dir=SMOKE_RESULTS_DIR,
        models_dir=MODELS_DIR,
        config=config,
    )
    done.add(key)

print(f'Smoke output folder: {SMOKE_RESULTS_DIR}')


## 7. Recap Smoke Results

If this table appears with valid metrics, Databricks can run the core PyNNLF workflow on this repo/project layout.


In [ ]:
import pynnlf

recap = pynnlf.recap_experiments(
    SMOKE_RESULTS_DIR,
    output_path=SMOKE_RESULTS_DIR / 'a1_experiment_result.csv',
    return_df=True,
)

cols = [
    'experiment_no', 'dataset_no', 'forecast_horizon_min', 'model_no',
    'hyperparameter_no', 'model_name', 'runtime_ms', 'test_nRMSE', 'test_nRMSE_stddev'
]
recap_display = recap.loc[
    recap['dataset_no'].astype(str).eq(DATASET_ID)
    & pd.to_numeric(recap['forecast_horizon_min'], errors='coerce').eq(forecast_horizon_min),
    cols,
].sort_values(['model_no', 'hyperparameter_no'])

display(recap_display)
print(f'Recap written to: {SMOKE_RESULTS_DIR / "a1_experiment_result.csv"}')


## 8. Interpretation

If `fast` passes, Databricks can read the publication data, import PyNNLF, run the engine, and write outputs. Then try `xgb` from the widget to check XGBoost. If you want to test the heaviest remaining paper dependencies, try `torch` and `prophet` separately.

The main unfinished paper experiment is the clean SA BESS 44-household block. Once the smoke tests pass, the next Databricks target is the missing `ds22`/`ds23`/`ds24` runs from `specs/sa_bess_44hh_batch.yaml`.
